## Dataset analysis: `student_transcript_list16.csv`

Purpose: **semester transcript** (List16). Contains target fields `SEMESTER_GPA` and **`CGPA`**.

This notebook profiles CGPA distribution/trends and checks key uniqueness (`REG_NO`, `SEMESTER_INDEX`).

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
DATA_DIR = Path.cwd()
df = pd.read_csv(DATA_DIR / "student_transcript_list16.csv")
df.shape

In [ ]:
df.head()

In [ ]:
key = ["REG_NO", "SEMESTER_INDEX"]
df.duplicated(key).sum(), df[key].isna().any(axis=1).sum()

In [ ]:
df[["SEMESTER_GPA", "CGPA"]].describe().T

In [ ]:
df.groupby("SEMESTER_INDEX")["CGPA"].agg(["count", "mean", "median", "std", "min", "max"]).reset_index().head(20)

## Advanced analytics

Focus: CGPA trajectories, volatility, and program/semester patterns (for feature design).

In [ ]:
from analysis_utils import basic_profile, missingness_report

print(basic_profile(df))
missingness_report(df, top_n=20)

In [ ]:
# CGPA trajectory features (per student)
df_sorted = df.sort_values(["REG_NO","SEMESTER_INDEX"]).copy()

traj = df_sorted.groupby("REG_NO").agg(
    semesters=("SEMESTER_INDEX","nunique"),
    cgpa_last=("CGPA","last"),
    cgpa_first=("CGPA","first"),
    cgpa_change=("CGPA", lambda s: float(s.iloc[-1] - s.iloc[0]) if len(s) > 1 else 0.0),
    gpa_std=("SEMESTER_GPA","std"),
)

traj.describe().T